<div dir="rtl">

# סבב 2: אימון multilingual-e5-large וחיפוש היברידי

המחברת הזאת ממשיכה את `hebrew_qa_retrieval.ipynb`, ועומדת בפני עצמה: היא טוענת את המאגר מחדש, ואת תוצאות הסבב הראשון מה-Dataset ב-Hugging Face.

מה השתנה לעומת הסבב הראשון, שבו האימון של e5-base שיפר פחות מנקודה אחת:
- **מודל בסיס גדול יותר:** `multilingual-e5-large` (560M פרמטרים), שכבר בלי אימון טוב ב-5 נקודות מ-e5-base.
- **2 שליליים קשים לכל שאלה**, וסינון מתון יותר. בסבב הראשון הסינון היה קשוח, ודילג על שאלות רבות.
- **חיפוש היברידי:** שילוב של BM25 עם המודל, כמו במערכות RAG אמיתיות.

**הרצה:** פתח את המחברת ב-Colab חדש עם `T4 GPU`, ואז `Runtime → Run all`. בערך שעה ורבע. בתא השלישי תתבקש להתחבר ל-Hugging Face, אז הישאר זמין בדקה הראשונה.

</div>

<div dir="rtl">

## 1. התקנה, הגדרות והתחברות

</div>

In [ ]:
!pip install -q -U sentence-transformers rank_bm25

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"   # less GPU memory fragmentation
import json, random, re, time, gc
import numpy as np, pandas as pd, torch, matplotlib.pyplot as plt
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device, torch.cuda.get_device_name(0) if device == "cuda" else "")

LARGE = "intfloat/multilingual-e5-large"
FT_L = "Fine-tuned e5-large (ours)"
FT_L_NAME = "hebrew-qa-retriever-e5-large"
DATASET_NAME = "hebrew-qa-retrieval-data"
EPOCHS_L, BATCH_L, MINI_L, LR_L, N_NEG, MAX_LEN = 1, 64, 8, 1e-5, 2, 384

In [ ]:
from huggingface_hub import login, HfApi, hf_hub_download
login()
api = HfApi()
user = api.whoami()["name"]
data_repo = f"{user}/{DATASET_NAME}"
print("Logged in as", user)

<div dir="rtl">

## 2. טעינת HeQ

</div>

In [ ]:
!git clone -q --depth 1 https://github.com/NNLP-IL/Hebrew-Question-Answering-Dataset heq 2>/dev/null || echo "already cloned"

corpus, para_id = [], {}
queries = {"train": [], "val": [], "test": []}
for split, fname in [("train", "train v1.1.json"), ("val", "val v1.1.json"), ("test", "test v1.1.json")]:
    data = json.load(open(f"heq/data/data v1.1/{fname}", encoding="utf-8"))["data"]
    for article in data:
        for p in article["paragraphs"]:
            text = p["context"].strip()
            if text not in para_id:
                para_id[text] = len(corpus)
                corpus.append({"id": len(corpus), "title": article["title"],
                               "source": article["source"], "split": split, "text": text})
            pid = para_id[text]
            for q in p["qas"]:
                if not q["is_impossible"]:
                    queries[split].append({"qid": q["id"], "question": q["question"].strip(),
                                           "pid": pid, "source": article["source"]})
corpus_texts = [p["text"] for p in corpus]
test_q = queries["test"]
pos = [q["pid"] for q in test_q]
print(f"Paragraphs: {len(corpus):,} | " + " | ".join(f"{s}: {len(v):,}" for s, v in queries.items()))

<div dir="rtl">

## 3. מדידה ותוצאות הסבב הקודם

אותה פונקציית מדידה כמו במחברת הראשונה. הטבלה מתחילה מהתוצאות שנשמרו ב-Hugging Face, ושורות חדשות מתווספות אליה.

</div>

In [ ]:
def metrics_from_ranks(ranks):
    r = np.asarray(ranks)
    return {"R@1": np.mean(r <= 1) * 100, "R@5": np.mean(r <= 5) * 100, "R@10": np.mean(r <= 10) * 100,
            "MRR@10": np.mean(np.where(r <= 10, 1 / r, 0)) * 100,
            "NDCG@10": np.mean(np.where(r <= 10, 1 / np.log2(r + 1), 0)) * 100}

def ranks_from_scores(scores, pos):
    pos = torch.as_tensor(pos, device=scores.device)
    pos_scores = scores[torch.arange(len(pos), device=scores.device), pos]
    return ((scores > pos_scores[:, None]).sum(1) + 1).cpu().numpy()

all_ranks = {}
all_results = pd.read_csv(hf_hub_download(data_repo, "results.csv", repo_type="dataset")).to_dict("records")

def report(name, ranks, qs):
    all_ranks[name] = ranks
    row = {"Model": name, **metrics_from_ranks(ranks)}
    for src in ["Wikipedia", "Geektime"]:
        mask = np.array([q["source"] == src for q in qs])
        row[f"R@5 {src}"] = metrics_from_ranks(ranks[mask])["R@5"]
    all_results[:] = [r for r in all_results if r["Model"] != name] + [row]
    display(pd.DataFrame(all_results).set_index("Model").round(1))

def encode(model, texts, kwargs, bs=64):
    return model.encode(texts, batch_size=bs, convert_to_tensor=True, normalize_embeddings=True,
                        show_progress_bar=True, **kwargs)

def free_gpu():
    gc.collect(); torch.cuda.empty_cache()

display(pd.DataFrame(all_results).set_index("Model").round(1))

<div dir="rtl">

## 4. BM25, e5-large בלי אימון, ובדיקה חוזרת של Qwen3

את BM25 ואת e5-large מחשבים מחדש, כי צריך אותם לחיפוש ההיברידי ולהשוואה. Qwen3 קיבל בסבב הראשון ציון נמוך באופן חשוד, אולי בגלל הרצה בדיוק מופחת (fp16), אז הפעם מריצים אותו בדיוק מלא.

</div>

In [ ]:
from rank_bm25 import BM25Okapi
tokenize = lambda t: re.findall(r"\w+", t.lower())
bm25 = BM25Okapi([tokenize(t) for t in corpus_texts])
bm25_scores = torch.tensor(np.array([bm25.get_scores(tokenize(q["question"])) for q in tqdm(test_q, desc="BM25")]))

E5 = dict(q=dict(prompt="query: "), d=dict(prompt="passage: "))
m = SentenceTransformer(LARGE, device=device).half()
m.max_seq_length = 512
C_large = encode(m, corpus_texts, E5["d"]).float().cpu()
Q_large_test = encode(m, [q["question"] for q in test_q], E5["q"]).float().cpu()
Q_large_train = encode(m, [q["question"] for q in queries["train"]], E5["q"]).float().cpu()
del m; free_gpu()
report("multilingual-e5-large", ranks_from_scores(Q_large_test @ C_large.T, pos), test_q)

m = SentenceTransformer("Qwen/Qwen3-Embedding-0.6B", device=device)   # full precision this time
m.max_seq_length = 512
C = encode(m, corpus_texts, {}, bs=32)
Q = encode(m, [q["question"] for q in test_q], dict(prompt_name="query"), bs=32)
report("Qwen3-Embedding-0.6B", ranks_from_scores(Q @ C.T, pos), test_q)
del m, C, Q; free_gpu()

<div dir="rtl">

## 5. שליליים קשים

לכל שאלת אימון, e5-large מחפש את הפסקאות הכי דומות, רק מתוך פסקאות האימון. מוותרים על:
- פסקאות מאותו ערך כמו הפסקה הנכונה, שעלולות להכיל את התשובה.
- פסקאות שקיבלו ציון גבוה מהפסקה הנכונה. גם הן חשודות כ"שליליים שקריים".

מהמועמדות שנשארו בוחרים באקראי 2 מתוך 15 הקשות ביותר. אף שאלה לא נזרקת: אם לא נמצאו מספיק מועמדות, לוקחים את הקשות ביותר בכל זאת.

</div>

In [ ]:
train_pids = [p["id"] for p in corpus if p["split"] == "train"]
train_index = {pid: j for j, pid in enumerate(train_pids)}
C_train = C_large[train_pids].to(device)
scores = Q_large_train.to(device) @ C_train.T
top_scores, top_idx = [t.cpu() for t in scores.topk(50, dim=1)]
pos_cols = torch.tensor([train_index[q["pid"]] for q in queries["train"]], device=device)
pos_scores = scores[torch.arange(len(pos_cols), device=device), pos_cols].cpu()
del C_train, scores; free_gpu()

train_rows, fallback = [], 0
for i, q in enumerate(tqdm(queries["train"], desc="Mining hard negatives")):
    title, below, above = corpus[q["pid"]]["title"], [], []
    for s, j in zip(top_scores[i].tolist(), top_idx[i].tolist()):
        pid = train_pids[j]
        if pid == q["pid"] or corpus[pid]["title"] == title:
            continue
        (below if s < float(pos_scores[i]) - 0.005 else above).append(pid)
    pool = below[:15]
    if len(pool) < N_NEG:
        fallback += 1
        pool = (below + above)[:15]
    row = {"anchor": "query: " + q["question"], "positive": "passage: " + corpus[q["pid"]]["text"]}
    for n, pid in enumerate(random.sample(pool, N_NEG), 1):
        row[f"negative_{n}"] = "passage: " + corpus[pid]["text"]
    train_rows.append(row)
print(f"Training rows: {len(train_rows):,} (none dropped; {fallback} used the fallback pool)")

<div dir="rtl">

## 6. אימון

כמו בסבב הראשון: `CachedMultipleNegativesRankingLoss`, בלי כפילויות ב-batch, ומדידה על ה-validation כל 100 צעדים. נקודת ההתחלה של e5-large על ה-validation היא Ndcg@10 של כ-0.806.

לוקח בערך 50 עד 60 דקות. השאר את הלשונית פתוחה.

</div>

In [ ]:
from datasets import Dataset
from sentence_transformers import SentenceTransformerTrainer, SentenceTransformerTrainingArguments
from sentence_transformers.losses import CachedMultipleNegativesRankingLoss
from sentence_transformers.training_args import BatchSamplers
from sentence_transformers.evaluation import InformationRetrievalEvaluator

val_evaluator = InformationRetrievalEvaluator(
    queries={q["qid"]: "query: " + q["question"] for q in queries["val"]},
    corpus={str(p["id"]): "passage: " + p["text"] for p in corpus if p["split"] != "test"},
    relevant_docs={q["qid"]: {str(q["pid"])} for q in queries["val"]},
    name="heq-val", mrr_at_k=[10], ndcg_at_k=[10], accuracy_at_k=[1, 5, 10], precision_recall_at_k=[10],
    show_progress_bar=False)

free_gpu()
print(f"GPU memory in use before training: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
model = SentenceTransformer(LARGE, device=device)
model.max_seq_length = MAX_LEN
args = SentenceTransformerTrainingArguments(
    output_dir="checkpoints-large", num_train_epochs=EPOCHS_L, per_device_train_batch_size=BATCH_L,
    learning_rate=LR_L, warmup_ratio=0.1, fp16=True, batch_sampler=BatchSamplers.NO_DUPLICATES,
    eval_strategy="steps", eval_steps=100, save_strategy="no", logging_steps=20,
    report_to="none", seed=SEED)
trainer = SentenceTransformerTrainer(
    model=model, args=args,
    train_dataset=Dataset.from_list(train_rows).shuffle(seed=SEED),
    loss=CachedMultipleNegativesRankingLoss(model, mini_batch_size=MINI_L),
    evaluator=val_evaluator)
start = time.time()
trainer.train()
print(f"Training took {(time.time() - start) / 60:.1f} minutes")
model.save("finetuned-large")

In [ ]:
# Back up the trained weights to Hugging Face right away
model.push_to_hub(f"{user}/{FT_L_NAME}", exist_ok=True)
print("Backed up:", f"https://huggingface.co/{user}/{FT_L_NAME}")

<div dir="rtl">

## 7. תוצאות וחיפוש היברידי

**חיפוש היברידי (Reciprocal Rank Fusion):** כל פסקה מקבלת ציון לפי המיקום שלה בשתי הרשימות: ‏1 חלקי (60 + המיקום ב-BM25), ועוד 1 חלקי (60 + המיקום במודל). פסקה שמופיעה גבוה בשתיהן עולה לראש. זו שיטה פשוטה שלא דורשת אימון, והיא מקובלת מאוד במערכות RAG.

</div>

In [ ]:
del trainer; free_gpu()
model.half()
model.max_seq_length = 512
C_ft = encode(model, corpus_texts, E5["d"]).float()
Q_ft = encode(model, [q["question"] for q in test_q], E5["q"]).float()
report(FT_L, ranks_from_scores(Q_ft @ C_ft.T, pos), test_q)

def to_ranks(S):
    order = S.argsort(dim=1, descending=True)
    r = torch.empty_like(order)
    r.scatter_(1, order, torch.arange(1, S.shape[1] + 1, device=S.device).expand_as(order))
    return r.float()

bm25_r = to_ranks(bm25_scores.to(device).float())
def hybrid(name, Q, C, k=60):
    dense_r = to_ranks(Q.to(device) @ C.to(device).T)
    report(name, ranks_from_scores(1 / (k + bm25_r) + 1 / (k + dense_r), pos), test_q)

hybrid("Hybrid: BM25 + e5-large (no training)", Q_large_test, C_large)
hybrid("Hybrid: BM25 + fine-tuned e5-large (ours)", Q_ft, C_ft)

In [ ]:
order = ["BM25 (keyword search)", "multilingual-e5-base", "multilingual-e5-large", "Qwen3-Embedding-0.6B",
         "Fine-tuned e5-base (ours)", FT_L,
         "Hybrid: BM25 + e5-large (no training)", "Hybrid: BM25 + fine-tuned e5-large (ours)"]
results = pd.DataFrame(all_results).set_index("Model")
results = results.loc[[m for m in order if m in results.index]].round(1)
results.to_csv("results.csv")
display(results)

ax = results[["R@1", "R@5", "R@10"]].plot.barh(figsize=(9, 7), width=0.8)
ax.set_xlabel("% of test questions where the correct paragraph was found")
ax.set_title("HeQ test set: retrieval accuracy")
ax.invert_yaxis(); ax.set_xlim(0, 100); ax.legend(loc="lower right")
for c in ax.containers: ax.bar_label(c, fmt="%.0f", fontsize=8)
plt.tight_layout(); plt.savefig("results.png", dpi=150); plt.show()

In [ ]:
base_r, ft_r = all_ranks["multilingual-e5-large"], all_ranks[FT_L]
improved = np.where((ft_r == 1) & (base_r > 1))[0]
worse = np.where((ft_r > 1) & (base_r == 1))[0]
print(f"e5-large -> fine-tuned: improved to rank 1: {len(improved)} | dropped from rank 1: {len(worse)}\n")
for idx, title in [(improved, "Improved"), (worse, "Got worse")]:
    print(f"=== {title} ===")
    for i in idx[:5]:
        q = test_q[i]
        print(f"Q: {q['question']}\n   base rank: {base_r[i]}  ->  fine-tuned rank: {ft_r[i]}")
        print(f"   paragraph: {corpus[q['pid']]['text'][:150]}...\n")

<div dir="rtl">

## 8. העלאה ל-Hugging Face

</div>

In [ ]:
model_repo = f"{user}/{FT_L_NAME}"
table = results.reset_index().to_markdown(index=False)
card = f'''---
language: [he]
license: mit
library_name: sentence-transformers
base_model: {LARGE}
datasets: [HeQ]
tags: [sentence-transformers, retrieval, hebrew, rag]
pipeline_tag: sentence-similarity
---

# Hebrew QA Retriever (fine-tuned multilingual-e5-large)

Retrieves the Hebrew paragraph that answers a Hebrew question. Fine-tuned from `{LARGE}` on the answerable
questions of [HeQ](https://github.com/NNLP-IL/Hebrew-Question-Answering-Dataset), with {N_NEG} mined hard
negatives per question and `CachedMultipleNegativesRankingLoss` (batch {BATCH_L}, {EPOCHS_L} epoch, lr {LR_L}).

## Usage

```python
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("{model_repo}")
q = model.encode("query: מי ניסח את מגילת העצמאות?", normalize_embeddings=True)
p = model.encode(["passage: ..."], normalize_embeddings=True)
```
Use the `query: ` and `passage: ` prefixes, as with the base model. For best results, combine it with
BM25 through Reciprocal Rank Fusion (the "Hybrid" rows below).

## Results (HeQ test set, {len(test_q):,} questions, search over all {len(corpus):,} paragraphs)

{table}

Test articles are disjoint from training articles. HeQ is licensed CC BY 4.0 (Webiks / MAFAT).
Limitation: HeQ questions were written while looking at the paragraph, so they share more wording
with it than real user queries would.
'''
api.upload_file(path_or_fileobj=card.encode(), path_in_repo="README.md", repo_id=model_repo)

os.makedirs("hub_data", exist_ok=True)
np.save("hub_data/emb_large.npy", C_large.numpy().astype(np.float16))
np.save("hub_data/emb_finetuned_large.npy", C_ft.cpu().numpy().astype(np.float16))
results.to_csv("hub_data/results.csv")
examples = [{"question": test_q[i]["question"], "pid": int(test_q[i]["pid"])}
            for i in list(improved[:6]) + random.sample(range(len(test_q)), 6)]
json.dump(examples, open("hub_data/examples.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)
api.upload_folder(folder_path="hub_data", repo_id=data_repo, repo_type="dataset")
print("Model:  ", f"https://huggingface.co/{model_repo}")
print("Dataset:", f"https://huggingface.co/datasets/{data_repo}")